In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import pynbody
from pynbody.analysis import profile

import os
import glob

from tqdm.notebook import tqdm


from cosmoSim import cosmoSim

In [2]:
matplotlib.rc('xtick', labelsize=12)
matplotlib.rc('ytick', labelsize=12)
matplotlib.rcParams['font.size']=12

In [3]:
data_path = "/home/ryan/Data/"
base_path="/home/ryan/projects/Medvedev/data_product_generation/data_products/data_prods/"

out_path = '../plots/811_HY/'


try:
    os.mkdir(out_path)
except:
    print(f'{out_path} already exists!')

../plots/811_HY/ already exists!


In [4]:
# z_max = 5

# with open('../../file_parts/ExpansionList_128', 'r') as f:
#     lines = f.read().split('\n')

# a = np.array([ float(l.strip().split(' ')[0]) for l in lines if l])

# zs = 1/a - 1

# zs = zs[ zs < z_max ]

In [5]:
CDM_RUNS = [
    'run_CDM_811_HY_dir_0'
]

tcDM_search = f'{data_path}/run_2cDM_811_HY_power00*Vkick*'

tcDM_RUNS = [ os.path.basename(x) for x in sorted(glob.glob(tcDM_search)) ]

In [6]:
Vkicks = [ cosmoSim(x, base_path=base_path).Vkick for x in tcDM_RUNS ]

c_norm = matplotlib.colors.LogNorm(vmin=np.amin(Vkicks), vmax=np.amax(Vkicks))
c_map = matplotlib.cm.plasma

s_map = matplotlib.cm.ScalarMappable(cmap=c_map, norm=c_norm)

In [7]:
soft_eff = 100000 / 2**11 / 39

In [ ]:
def calculate_profile(run_name, redshift, halo_index=0):
    
    run = cosmoSim(run_name, base_path=base_path)
    idx = run.redshift_to_index(redshift)
    # load the file
    # print(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    f = pynbody.load(data_path + f'/{run_name}/snap_{idx:03}.hdf5')
    h = f.halos(subhalos=True)

    f.physical_units()
    f['rho']; f['smooth'] # force KDTree to be built and density estimations made
    # only care about most massive halo for now
    pynbody.analysis.faceon(h[halo_index])
    min_rad = np.amin([ 0.1 * soft_eff, 0.1 * h[halo_index].properties['SubhaloHalfmassRad']])
    max_rad = 3 * h[halo_index].properties['SubhaloHalfmassRad']
    p_3d = profile.Profile(f, rmin = min_rad, rmax = max_rad, ndim = 3)
    return p_3d['rbins'].in_units('kpc'), p_3d['density'].in_units('Msol kpc^-3'), h[halo_index].properties['SubhaloMassInRad']

In [ ]:
redshift=0
for halo_idx in range(3):
    fig, ax = plt.subplots(1, 1, figsize=[12, 12])

    runf = cosmoSim(CDM_RUNS[0], base_path=base_path)
    idx = runf.redshift_to_index(redshift)

    # diagnostic only
    # if idx < 127:
    #     plt.close(fig)
    #     continue

    # plot 2cDM runs
    for rn in tqdm(tcDM_RUNS):
        run = cosmoSim(rn, base_path=base_path)

        # print(idx, rn)
        try:
            rbins, masses, _ = calculate_profile(rn, redshift, halo_index=halo_idx)
        except KeyError:
            print(f'Error with run {rn} for profile {idx}')
            print('Skipping...')
            continue

        plt.plot(rbins, masses, color=s_map.to_rgba(run.Vkick))

    fig.colorbar(s_map, label='$V_{kick}$ [km s$^{-1}$]', ax=ax)

    # plot CDM run
    for rn in CDM_RUNS:

        rbins, masses, halo_mass = calculate_profile(rn, redshift, halo_index=halo_idx)

        plt.plot(rbins, masses, color='k', linestyle=(5, (10, 3)))

    # plot formatting
    ax.set_xlabel('$r$ [kpc]')
    ax.set_ylabel(r'$\rho$ [M$_{\odot}$ kpc$^{-3}$]')

    ax.vlines(soft_eff, 10**2, 10**50, colors='k', linestyles='dashed')
    ax.vlines(2.8*soft_eff, 10**2, 10**50, colors='k', linestyles='dashdot')

    ylims = [
        1/2.8 * np.amin( masses[ np.nonzero(masses) ] ),
        2.8 * np.amax( masses[ np.nonzero(masses) ] )
    ]
    ax.set_ylim(ylims)

    ax.plot([],[], label=f'z = {redshift:.3f}', alpha=0)
    ax.plot([],[], label=f'Halo Mass = {halo_mass:.3e}', alpha=0)
    

    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.grid(True, which="both", ls="-")
    ax.set_box_aspect(aspect=1)
    ax.legend()
    plt.title('(0, 0) Model')

    fname = f'halo{halo_idx}_811_HY_00_{idx:03}.png'
    plt.savefig(out_path + fname)

    plt.close(fig)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [10]:
CDM_RUNS = [
    'run_CDM_811_HY_dir_0'
]

tcDM_search = f'{data_path}/run_2cDM_811_HY_powerm2m2*Vkick*'

tcDM_RUNS = [ os.path.basename(x) for x in sorted(glob.glob(tcDM_search)) ]

Vkicks = [ cosmoSim(x, base_path=base_path).Vkick for x in tcDM_RUNS ]

c_norm = matplotlib.colors.LogNorm(vmin=np.amin(Vkicks), vmax=np.amax(Vkicks))
c_map = matplotlib.cm.plasma

s_map = matplotlib.cm.ScalarMappable(cmap=c_map, norm=c_norm)

In [ ]:
redshift=0
for halo_idx in range(3):
    fig, ax = plt.subplots(1, 1, figsize=[12, 12])

    runf = cosmoSim(CDM_RUNS[0], base_path=base_path)
    idx = runf.redshift_to_index(redshift)

    # diagnostic only
    # if idx < 127:
    #     plt.close(fig)
    #     continue

    # plot 2cDM runs
    for rn in tqdm(tcDM_RUNS):
        run = cosmoSim(rn, base_path=base_path)

        # print(idx, rn)
        try:
            rbins, masses, _ = calculate_profile(rn, redshift, halo_index=halo_idx)
        except KeyError:
            print(f'Error with run {rn} for profile {idx}')
            print('Skipping...')
            continue

        plt.plot(rbins, masses, color=s_map.to_rgba(run.Vkick))

    fig.colorbar(s_map, label='$V_{kick}$ [km s$^{-1}$]', ax=ax)

    # plot CDM run
    for rn in CDM_RUNS:

        rbins, masses, halo_mass = calculate_profile(rn, redshift, halo_index=halo_idx)

        plt.plot(rbins, masses, color='k', linestyle=(5, (10, 3)))

    # plot formatting
    ax.set_xlabel('$r$ [kpc]')
    ax.set_ylabel(r'$\rho$ [M$_{\odot}$ kpc$^{-3}$]')

    ax.vlines(soft_eff, 10**2, 10**50, colors='k', linestyles='dashed')
    ax.vlines(2.8*soft_eff, 10**2, 10**50, colors='k', linestyles='dashdot')

    ylims = [
        1/2.8 * np.amin( masses[ np.nonzero(masses) ] ),
        2.8 * np.amax( masses[ np.nonzero(masses) ] )
    ]
    ax.set_ylim(ylims)

    ax.plot([],[], label=f'z = {redshift:.3f}', alpha=0)
    ax.plot([],[], label=f'Halo Mass = {halo_mass:.3e}', alpha=0)
    

    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.grid(True, which="both", ls="-")
    ax.set_box_aspect(aspect=1)
    ax.legend()
    plt.title('(-2, -2) Model')

    fname = f'halo{halo_idx}_811_HY_mm_{idx:03}.png'
    plt.savefig(out_path + fname)

    plt.close(fig)

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]